In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report
from torch.optim import lr_scheduler

import numpy as np
import os
import logging
import matplotlib.pyplot as plt
import seaborn as sns

# =====================
# CẤU HÌNH HỆ THỐNG
# =====================
BATCH_SIZE = 8 
IMG_SIZE = (256, 256) # Bài báo sử dụng 256x256 
SEED = 42 
torch.manual_seed(SEED)

BASE_INPUT_PATH = "/kaggle/input/datasets/bananalatraichuoi/rambutan-uiters"
WORKING_PATH = "/kaggle/working/results"
os.makedirs(WORKING_PATH, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Convolutional Block Attention Module (CBAM) ---
# Giúp mạng tập trung vào các đặc trưng quan trọng theo kênh và không gian [cite: 132, 244]
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        # Channel Attention
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid_channel = nn.Sigmoid()
        
        # Spatial Attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        # Channel Attention [cite: 254-256]
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        out = self.sigmoid_channel(avg_out + max_out).view(b, c, 1, 1)
        x = x * out
        
        # Spatial Attention [cite: 257-259]
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial_out = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * spatial_out

# --- 2. Inception-ResNet Block ---
# Trích xuất đặc trưng ở nhiều quy mô khác nhau với chi phí tính toán thấp [cite: 131, 238]
class InceptionResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(InceptionResNetBlock, self).__init__()
        self.branch1x1 = nn.Conv2d(in_channels, out_channels // 4, kernel_size=1)
        
        self.branch3x3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1),
            nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=3, padding=1)
        )
        
        self.branch5x5 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1),
            nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=3, padding=1),
            nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=3, padding=1)
        )
        
        self.reduction = nn.Conv2d(out_channels // 4 * 3, out_channels, kernel_size=1)
        self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = self.shortcut(x)
        b1 = self.branch1x1(x)
        b2 = self.branch3x3(x)
        b3 = self.branch5x5(x)
        out = torch.cat([b1, b2, b3], dim=1)
        out = self.reduction(out)
        return self.relu(out + res)

# --- 3. Coordinate Attention (CA) ---
# Nắm bắt thông tin vị trí chính xác trên trục X và Y [cite: 122, 273]
class CoordinateAttention(nn.Module):
    def __init__(self, in_channels, out_channels, reduction=32):
        super(CoordinateAttention, self).__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        
        mip = max(8, in_channels // reduction)
        self.conv1 = nn.Conv2d(in_channels, mip, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.ReLU(inplace=True)
        
        self.conv_h = nn.Conv2d(mip, out_channels, kernel_size=1)
        self.conv_w = nn.Conv2d(mip, out_channels, kernel_size=1)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        
        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))
        
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        
        a_h = torch.sigmoid(self.conv_h(x_h))
        a_w = torch.sigmoid(self.conv_w(x_w))
        
        return identity * a_h * a_w

# --- 4. GrapeLeafNet Main Architecture ---
class GrapeLeafNet(nn.Module):
    def __init__(self, num_classes):
        super(GrapeLeafNet, self).__init__()
        
        # Track 1: CNN Branch [cite: 118, 130]
        self.cnn_track = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            InceptionResNetBlock(32, 64),
            nn.MaxPool2d(2),
            CBAM(64),
            InceptionResNetBlock(64, 128),
            nn.MaxPool2d(2),
            CBAM(128)
        )
        
        # Track 2: Transformer Branch (Simplified) [cite: 133]
        self.patch_embed = nn.Conv2d(3, 128, kernel_size=4, stride=4)
        encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # Fusion & Classification [cite: 134]
        self.ca = CoordinateAttention(256, 256)
        self.flatten = nn.Flatten()
        
        # Tính toán Linear input: 256 channels * (256/4 / 2 / 2)^2 = 256 * 16 * 16 (nếu maxpool x2)
        # Để an toàn, ta dùng AdaptiveAvgPool2d để cố định kích thước trước khi vào Linear
        self.global_pool = nn.AdaptiveAvgPool2d((7, 7)) 
        self.fc = nn.Sequential(
            nn.Linear(256 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # Track 1: [B, 128, 64, 64] (sau 2 lần maxpool từ 256)
        x1 = self.cnn_track(x)
        
        # Track 2: Transformer
        x2 = self.patch_embed(x) # [B, 128, 64, 64]
        b, c, h, w = x2.shape
        x2_flat = x2.flatten(2).permute(0, 2, 1) # [B, Seq, C]
        x2_trans = self.transformer_encoder(x2_flat)
        x2 = x2_trans.permute(0, 2, 1).view(b, c, h, w)
        
        # Fusion [cite: 134]
        out = torch.cat([x1, x2], dim=1) # [B, 256, 64, 64]
        out = self.ca(out)
        out = self.global_pool(out)
        out = self.flatten(out)
        return self.fc(out)
        
class MapDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.dataset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.dataset)
        
# --- 5. DATA PIPELINE ---
def get_dataloaders():
    # Dataset gốc không để transform
    full_dataset = datasets.ImageFolder(BASE_INPUT_PATH)
    class_names = full_dataset.classes
    # Chia tỉ lệ 60:20:20 [cite: 324]
    train_size = int(0.6 * len(full_dataset))
    val_size = int(0.2 * len(full_dataset))
    test_size = len(full_dataset) - train_size - val_size
    
    train_subset, val_subset, test_subset = random_split(
        full_dataset, [train_size, val_size, test_size], 
        generator=torch.Generator().manual_seed(SEED)
    )

    # --- TÍNH TRỌNG SỐ TỰ ĐỘNG Ở ĐÂY ---
    # Lấy các nhãn (targets) của chỉ riêng tập Train
    train_targets = [full_dataset.targets[i] for i in train_subset.indices]
    from collections import Counter
    counts = Counter(train_targets)
    # Đảm bảo thứ tự khớp với class_names
    class_counts = [counts[i] for i in range(len(class_names))]
    
    weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    weights = (weights / weights.sum()).to(device)
    # ----------------------------------
    
    # Định nghĩa Transform chuẩn theo bài báo [cite: 318, 323]
    train_transform = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.RandomRotation(40), # 
        transforms.RandomHorizontalFlip(), # 
        transforms.RandomVerticalFlip(), # 
        transforms.ColorJitter(brightness=(0.5, 1.5)), # 
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8)), # 
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    test_transform = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Áp dụng Transform riêng biệt thông qua MapDataset
    train_data = MapDataset(train_subset, transform=train_transform)
    val_data = MapDataset(val_subset, transform=test_transform)
    test_data = MapDataset(test_subset, transform=test_transform)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)
    
    return train_loader, val_loader, test_loader, full_dataset.classes, weights

# --- 6. CHI TIẾT ĐÁNH GIÁ (ACCURACY TỪNG CLASS) ---
def evaluate_detailed_performance(model, test_loader, class_names):
    model.eval()
    y_true = []
    y_pred = []
    
    logger.info("🔮 Đang dự đoán trên tập dữ liệu kiểm thử...")
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # Báo cáo chi tiết từng class [cite: 331]
    print("\n📊 BÁO CÁO PHÂN LOẠI CHI TIẾT:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix - Đồ án Chôm chôm/Ớt')
    plt.ylabel('Thực tế')
    plt.xlabel('Dự đoán')
    plt.savefig(os.path.join(WORKING_PATH, 'confusion_matrix.png'))
    plt.show()

# 3. Vòng lặp huấn luyện chi tiết
def train_model(model, train_loader, val_loader, epochs=50):
    best_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        train_acc = 100 * correct / total
        
        # Validation Phase
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        print(f'Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(train_loader):.4f} - Train Acc: {train_acc:.2f}% - Val Acc: {val_acc:.2f}%')
        
        # Lưu lại model tốt nhất
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_model_rambutan.pth')
            print("--> Đã lưu mô hình có Val Acc tốt nhất!")

        # 3.2. Lưu checkpoint ĐỊNH KỲ (Ví dụ mỗi 20 Epoch lưu 1 lần) [cite: 362, 531]
        # Mục đích: Phòng trường hợp Kaggle bị ngắt session giữa chừng.
        if (epoch + 1) % 20 == 0:
            checkpoint_periodic = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
            }
            periodic_path = os.path.join(WORKING_PATH, f'checkpoint_epoch_{epoch+1}.pth')
            torch.save(checkpoint_periodic, periodic_path)
            print(f"--> [PERIODIC] Đã lưu checkpoint định kỳ tại {periodic_path}")


if __name__ == "__main__":
    # Khởi tạo dữ liệu và lấy trọng số đã tính tự động
    train_loader, val_loader, test_loader, class_names, class_weights = get_dataloaders()
    
    # 1. Khởi tạo mô hình [cite: 18]
    model = GrapeLeafNet(num_classes=len(class_names)).to(device)
    
    # 2. Cấu hình Loss và Optimizer theo đúng bài báo [cite: 328]
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0001)
    
    # Thêm Scheduler để giảm LR khi gặp cao nguyên (plateau)
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)
    
    # 3. Huấn luyện 120 Epoch [cite: 364]
    # Lưu ý: Trong hàm train_model, hãy nhớ gọi scheduler.step(val_loss)
    train_model(model, train_loader, val_loader, epochs=120)
    
    # 4. Đánh giá cuối cùng với các chỉ số Accuracy, Precision, Recall, F1 [cite: 331]
    logger.info("📂 Đang kiểm tra hiệu suất trên tập Test...")
    model.load_state_dict(torch.load('best_model_rambutan.pth'))
    evaluate_detailed_performance(model, test_loader, class_names)